In [11]:
import yaml
from ssm import *

model = SSMTranslator(config=SSMTranslatorConfig(**yaml.safe_load(open("../configs/ssm_tuning/model.yaml", "r"))))
state_dict = { x[len("module."):]: v for x, v in torch.load("../temp/best_model.pt").items() }
model.load_state_dict(state_dict)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

SSMTranslator(
  (E): Embedding(32000, 768)
  (encoder_layers): ModuleList(
    (0-4): 5 x Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(2304, 2304, kernel_size=(4,), stride=(1,), groups=2304)
      (lin_gate): MultiheadLinear()
      (norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (lin_out): Linear(in_features=768, out_features=768, bias=True)
    )
    (5): Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(1536, 1536, kernel_size=(4,), stride=(1,), groups=1536)
    )
  )
  (lin_EDs): ModuleList(
    (0-5): 6 x Linear(in_features=64, out_features=64, bias=True)
  )
  (decoder_layers): ModuleList(
    (0-5): 6 x Mamba2Layer(
      (lin_logA): MultiheadLinear()
      (lin_XBC): MultiheadLinear()
      (conv): Conv1d(2304, 2304, kernel_size=(4,), stride=(1,), groups=2304)
      (lin_gate): MultiheadLinear()
      (norm): LayerNorm((768,), eps=1e-05, elemen

In [ ]:
import sentencepiece as sp

proc_en = sp.SentencePieceProcessor()
proc_fr = sp.SentencePieceProcessor()

proc_en.Load("../vocab/en.model")
proc_fr.Load("../vocab/fr.model")

sentence = "Hello, how are you today?"
en_ids = proc_en.Encode(sentence, add_bos=True, add_eos=True)
en_ids = torch.tensor(en_ids).unsqueeze(0).to(device)

fr_ids = proc_fr.Encode("Bonjour, comment ça va aujourd'hui?", add_bos=True, add_eos=True)
fr_ids = torch.tensor(fr_ids).unsqueeze(0).to(device)

print(en_ids[:, :-1])
print(fr_ids[:, 1:])

logits = model(
    inp_ids=en_ids[:, :-1],
    decode_method="forced",
    max_output_len=50,
    forcing_ids=fr_ids[:, 1:],
)

ids = torch.argmax(logits, dim=-1).squeeze(0).tolist()

print(proc_fr.Decode(ids))

Bonjour, comment ça va aujourd'hui?


In [22]:
ids.shape

torch.Size([])